# Chapter 14: Inheritance: For Better or for Worse
**From: Fluent Python, 2nd Edition**

## TL;DR
- **super()** returns a dynamic proxy that delegates to the next class in the MRO, not just "the parent class"
- **Never subclass dict/list/str directly** — their C implementations ignore your overrides; use UserDict/UserList/UserString instead
- **MRO (Method Resolution Order)** uses C3 linearization to produce a deterministic search order stored in `__mro__`
- **Mixin classes** provide reusable method bundles via multiple inheritance without defining standalone types
- **Cooperative methods** call `super()` so the entire MRO chain is traversed; one missing `super()` call breaks the chain
- **Best practice**: favor composition over inheritance, use ABCs for interfaces, name mixins with a `…Mixin` suffix

---
## 1. The super() Function

`super()` returns a proxy object that delegates method calls to the next class in the **Method Resolution Order** (MRO). In Python 3 the compiler fills in the arguments automatically.

In [ ]:
from collections import OrderedDict

class LastUpdatedOrderedDict(OrderedDict):
    """Store items in the order they were last updated."""

    def __setitem__(self, key, value):
        super().__setitem__(key, value)   # delegates to OrderedDict
        self.move_to_end(key)

d = LastUpdatedOrderedDict()
d['a'] = 1
d['b'] = 2
d['a'] = 10   # 'a' moves to the end
print(list(d.items()))  # [('b', 2), ('a', 10)]

### Why not call the superclass by name?

Hardcoding the parent name works but is fragile — if the base class changes, you have to update every call site. And it breaks cooperative multiple inheritance.

In [ ]:
# Anti-pattern: hardcoded superclass name
class NotRecommended(OrderedDict):
    def __setitem__(self, key, value):
        OrderedDict.__setitem__(self, key, value)  # fragile!
        self.move_to_end(key)

# Works, but if you later change the base class to something else,
# you must remember to update this line too.
nr = NotRecommended()
nr['x'] = 1
print(nr)

### super() with explicit arguments (Python 2 style, still valid)

`super(ClassName, self)` — the compiler does this for you in Python 3.

In [ ]:
class ExplicitSuper(OrderedDict):
    def __setitem__(self, key, value):
        # Equivalent to super().__setitem__(key, value) in Py3
        super(ExplicitSuper, self).__setitem__(key, value)
        self.move_to_end(key)

es = ExplicitSuper(a=1)
es['b'] = 2
print(es)

---
## 2. Subclassing Built-In Types Is Tricky

Built-in types (`dict`, `list`, `str`) are implemented in C, and their internal methods **do not** call overridden methods in Python subclasses. This violates the principle of late binding.

In [ ]:
# PROBLEM: __setitem__ override is ignored by dict.__init__ and dict.update

class DoppelDict(dict):
    """Doubles every value on storage — or does it?"""
    def __setitem__(self, key, value):
        super().__setitem__(key, [value] * 2)

dd = DoppelDict(one=1)
print("After init:", dd)            # {'one': 1} — NOT doubled!

dd['two'] = 2
print("After []=:", dd)             # 'two': [2, 2] — doubled (direct call)

dd.update(three=3)
print("After update:", dd)          # 'three': 3 — NOT doubled!

### The fix: use collections.UserDict

`UserDict` is written in Python, so it properly calls your overrides.

In [ ]:
import collections

class DoppelDict2(collections.UserDict):
    """Doubles every value on storage — correctly this time."""
    def __setitem__(self, key, value):
        super().__setitem__(key, [value] * 2)

dd2 = DoppelDict2(one=1)
print("After init:", dd2)            # {'one': [1, 1]} — doubled!

dd2['two'] = 2
print("After []=:", dd2)             # 'two': [2, 2] — doubled!

dd2.update(three=3)
print("After update:", dd2)          # 'three': [3, 3] — doubled!

In [ ]:
# Another example: AnswerDict always returns 42 from __getitem__

class AnswerDict(dict):
    def __getitem__(self, key):
        return 42

ad = AnswerDict(a='foo')
print("ad['a']:", ad['a'])           # 42 — our override works for direct access

d = {}
d.update(ad)                          # dict.update ignores our __getitem__
print("d['a']:", d['a'])             # 'foo' — NOT 42!

# Fix: use UserDict
class AnswerDict2(collections.UserDict):
    def __getitem__(self, key):
        return 42

ad2 = AnswerDict2(a='foo')
d2 = {}
d2.update(ad2)
print("d2['a']:", d2['a'])           # 42 — correct!

---
## 3. Multiple Inheritance and Method Resolution Order (MRO)

When a class has multiple base classes, Python uses the **C3 linearization** algorithm to compute a deterministic search order — the **MRO** — stored in `__mro__`.

In [ ]:
# The classic "diamond problem"

class Root:
    def ping(self):
        print(f"  {type(self).__name__}.ping() in Root")

    def pong(self):
        print(f"  {type(self).__name__}.pong() in Root")

    def __repr__(self):
        return f"<instance of {type(self).__name__}>"

class A(Root):
    def ping(self):
        print(f"  {type(self).__name__}.ping() in A")
        super().ping()       # cooperates!

    def pong(self):
        print(f"  {type(self).__name__}.pong() in A")
        super().pong()       # cooperates!

class B(Root):
    def ping(self):
        print(f"  {type(self).__name__}.ping() in B")
        super().ping()       # cooperates!

    def pong(self):
        print(f"  {type(self).__name__}.pong() in B")
        # Does NOT call super().pong() — breaks the chain!

class Leaf(A, B):
    def ping(self):
        print(f"  {type(self).__name__}.ping() in Leaf")
        super().ping()

leaf = Leaf()
print("--- leaf.ping() ---")
leaf.ping()
# Full chain: Leaf -> A -> B -> Root (all cooperate)

print()
print("--- leaf.pong() ---")
leaf.pong()
# Chain: A -> B (B does not call super, so Root.pong is skipped)

In [ ]:
# Inspect the MRO directly
print("Leaf MRO:")
for cls in Leaf.__mro__:
    print(f"  {cls.__name__}")

### Key insight: MRO depends on declaration order

If `Leaf` were `Leaf(B, A)` instead of `Leaf(A, B)`, `B` would come before `A` in the MRO.

In [ ]:
class Leaf2(B, A):
    def ping(self):
        print(f"  {type(self).__name__}.ping() in Leaf2")
        super().ping()

print("Leaf2 MRO:", [c.__name__ for c in Leaf2.__mro__])

print()
print("--- leaf2.ping() ---")
Leaf2().ping()

print()
print("--- leaf2.pong() ---")
Leaf2().pong()
# B.pong doesn't call super(), so A.pong and Root.pong are skipped!

---
## 4. Mixin Classes

A **mixin** is a class designed to be combined with other classes via multiple inheritance. It adds or customizes behavior but is **not** meant to stand alone. By convention, name them with a `Mixin` suffix.

In [ ]:
import collections

def _upper(key):
    """Uppercase a key if it's a string, otherwise return unchanged."""
    try:
        return key.upper()
    except AttributeError:
        return key

class UpperCaseMixin:
    """Mixin that uppercases string keys in a mapping."""

    def __setitem__(self, key, item):
        super().__setitem__(_upper(key), item)

    def __getitem__(self, key):
        return super().__getitem__(_upper(key))

    def get(self, key, default=None):
        return super().get(_upper(key), default)

    def __contains__(self, key):
        return super().__contains__(_upper(key))

# Combine the mixin with UserDict — mixin MUST come first
class UpperDict(UpperCaseMixin, collections.UserDict):
    pass

d = UpperDict([('a', 'letter A'), (2, 'digit two')])
print("Keys:", list(d.keys()))      # ['A', 2]

d['b'] = 'letter B'
print("'b' in d:", 'b' in d)        # True (looks up 'B')
print("d['a']:", d['a'])             # 'letter A'
print("d.get('B'):", d.get('B'))     # 'letter B'
print("All keys:", list(d.keys()))   # ['A', 2, 'B']

In [ ]:
# The same mixin works with Counter too!
class UpperCounter(UpperCaseMixin, collections.Counter):
    """Counter that uppercases string keys."""

c = UpperCounter('BaNanA')
print("Most common:", c.most_common())  # [('A', 3), ('N', 2), ('B', 1)]

### Why the mixin must come first in the base list

If you wrote `class UpperDict(collections.UserDict, UpperCaseMixin)`, the `UserDict` methods would be found first in the MRO, and the mixin's methods would never be called.

---
## 5. Cooperative Multiple Inheritance

For multiple inheritance to work reliably, **every** method in the chain must call `super()`. This is called **cooperative** behavior.

In [ ]:
# Demonstrate: super() behavior depends on the RUNTIME MRO, not the class definition

class U:
    """A class unrelated to Root, A, or B."""
    def ping(self):
        print(f"  {type(self).__name__}.ping() in U")
        super().ping()   # At definition time, U's super is object (which has no ping!)

# U().ping()  # This would raise AttributeError!

# But when combined in a multiple-inheritance hierarchy...
class LeafUA(U, A):
    def ping(self):
        print(f"  {type(self).__name__}.ping() in LeafUA")
        super().ping()

print("LeafUA MRO:", [c.__name__ for c in LeafUA.__mro__])
print()
print("--- LeafUA().ping() ---")
LeafUA().ping()
# LeafUA -> U -> A -> Root -> object
# U.ping()'s super().ping() now reaches A.ping(), not object!

### The cooperative contract
- Every non-root method `m` should call `super().m()`
- Cooperative methods must have **compatible signatures**
- The **activation sequence** depends on both the MRO and whether each method calls `super()`
- A single non-cooperative method breaks the chain silently

---
## 6. Real-World Multiple Inheritance Examples

### ABCs as Mixins
`collections.abc.MutableMapping` provides mixin methods like `update`, `pop`, `setdefault` that rely on the abstract methods you implement.

### ThreadingMixIn (socketserver)
```python
# From the standard library — this is the ENTIRE class!
class ThreadingHTTPServer(socketserver.ThreadingMixIn, HTTPServer):
    daemon_threads = True
```

### Django Generic Views
Django's class-based views use mixins extensively with explicit naming:
- `TemplateResponseMixin` — renders templates
- `MultipleObjectMixin` — provides queryset iteration + pagination
- `ListView` — an *aggregate class* combining mixins (body is just a docstring)

### Tkinter (cautionary tale)
- Deep hierarchies with 4+ levels
- `Misc` class has 100+ methods inherited by every widget
- Geometry managers (`Pack`, `Place`, `Grid`) should use composition, not inheritance

---
## 7. Coping with Inheritance: Best Practices

In [ ]:
# Best Practice 1: Favor composition over inheritance

class Engine:
    def start(self):
        return "Engine started"

# BAD: Car IS-A Engine? No!
class BadCar(Engine):
    pass

# GOOD: Car HAS-A Engine (composition)
class GoodCar:
    def __init__(self):
        self._engine = Engine()

    def start(self):
        return self._engine.start()

car = GoodCar()
print(car.start())

In [ ]:
# Best Practice 2: Use ABCs for interface inheritance
from abc import ABC, abstractmethod

class Serializable(ABC):
    @abstractmethod
    def serialize(self) -> bytes:
        ...

class JsonMixin:
    """Mixin that adds JSON serialization (named with Mixin suffix)."""
    def to_json(self) -> str:
        import json
        return json.dumps(self.__dict__)

# Best Practice 3: Explicit mixin + ABC
class UserRecord(JsonMixin, Serializable):
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def serialize(self) -> bytes:
        return self.to_json().encode('utf-8')

user = UserRecord("Alice", 30)
print("JSON:", user.to_json())
print("Bytes:", user.serialize())

In [ ]:
# Best Practice 4: Provide aggregate classes
# An aggregate class combines mixins so users don't have to

class LoggingMixin:
    def log(self, msg):
        print(f"[{type(self).__name__}] {msg}")

class ValidatingMixin:
    def validate(self):
        for attr, value in self.__dict__.items():
            if value is None:
                raise ValueError(f"{attr} cannot be None")
        return True

# Aggregate class — the user just inherits this one class
class RobustRecord(LoggingMixin, ValidatingMixin, Serializable):
    """Aggregate: logging + validation + serializable interface."""
    def __init__(self, name, value):
        self.name = name
        self.value = value

    def serialize(self) -> bytes:
        return f"{self.name}={self.value}".encode()

r = RobustRecord("temp", 98.6)
r.log("Created")
r.validate()
print(r.serialize())

### Summary of Best Practices

| Guideline | Rationale |
|-----------|-----------|
| Favor composition over inheritance | Reduces coupling, more flexible |
| Use ABCs for interfaces | Makes "is-a" relationships explicit |
| Name mixins with `...Mixin` suffix | Signals intent, prevents misuse |
| Provide aggregate classes | Saves users from assembling mixins |
| Subclass only classes designed for it | Prevents fragile override bugs |
| Avoid subclassing concrete classes | Internal state easily corrupted |
| Every method should call `super()` | Ensures cooperative chain works |

---
## Try It Yourself

In [ ]:
# Exercise 1: Fix this broken subclass by using UserList instead of list
# Currently, __init__ doesn't call our __setitem__

import collections

# TODO: change the base class to fix the behavior
class DoubleList(collections.UserList):
    """A list that stores each item twice."""
    def __setitem__(self, index, item):
        super().__setitem__(index, item)
        # For simplicity, just double on append
    def append(self, item):
        super().append(item)
        super().append(item)

dl = DoubleList()
dl.append('hello')
print(dl)  # Should contain 'hello' twice: ['hello', 'hello']

In [ ]:
# Exercise 2: Create a TimestampMixin that records when items are set

import time

class TimestampMixin:
    """Mixin that records the timestamp when items are added to a mapping."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._timestamps = {}

    def __setitem__(self, key, value):
        super().__setitem__(key, value)
        self._timestamps[key] = time.time()

    def get_timestamp(self, key):
        return self._timestamps.get(key)

class TimestampDict(TimestampMixin, collections.UserDict):
    pass

td = TimestampDict()
td['x'] = 1
td['y'] = 2
time.sleep(0.01)
td['x'] = 10  # update

print("Data:", dict(td))
print("x was last set at:", td.get_timestamp('x'))
print("y was last set at:", td.get_timestamp('y'))
print("x timestamp > y timestamp:", td.get_timestamp('x') > td.get_timestamp('y'))

In [ ]:
# Exercise 3: Predict the MRO
# Before running, guess what W.__mro__ will be

class X:
    pass

class Y(X):
    pass

class Z(X):
    pass

class W(Y, Z):
    pass

# Predict, then verify:
print("W MRO:", [c.__name__ for c in W.__mro__])
# Answer: W -> Y -> Z -> X -> object
# (C3 linearization keeps the declaration order Y, Z
#  and defers the common ancestor X until after both)

---
## Key Takeaways

1. **Always use `super()`** instead of hardcoding parent class names — it enables cooperative multiple inheritance
2. **Never subclass `dict`, `list`, or `str` directly** — use `UserDict`, `UserList`, `UserString` from `collections`
3. **The MRO (`__mro__`)** determines method lookup order; inspect it to debug inheritance issues
4. **Mixin classes** provide reusable behavior bundles — name them with `Mixin` suffix, place them first in the base class list
5. **Cooperative methods** must all call `super()` — one missing call silently breaks the chain
6. **Favor composition over inheritance** — it's more flexible and easier to understand
7. **Use ABCs for interfaces, mixins for code reuse** — keep the two concerns separate

## See Also

- [[super-function]] — deep dive on super() mechanics
- [[subclassing-built-in-types]] — why dict/list/str subclasses misbehave
- [[multiple-inheritance-mro]] — C3 linearization and __mro__
- [[mixin-classes]] — designing reusable mixins
- [[cooperative-multiple-inheritance]] — the cooperative contract
- [[real-world-multiple-inheritance]] — Django, Tkinter, socketserver examples
- [[inheritance-best-practices]] — the seven rules for coping with inheritance